In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostRegressor


# ==========================================
# 1. Read Data
# ==========================================
df = pd.read_excel('Final_Combined_Properties.xlsx')

# ==========================================
# 3. Clean placeholders
# ==========================================
df.replace(['Not Found', 'No Description'], np.nan, inplace=True)

# ==========================================
# 4. Drop unused columns
# ==========================================
df.drop(columns=['URL','Image_URL','Title','Description','Price/m²'],
        inplace=True, errors='ignore')

# ==========================================
# 5. Remove duplicates
# ==========================================
df.drop_duplicates(inplace=True)

# ==========================================
# 6. Convert numeric safely
# ==========================================
df['Floor'] = df['Floor'].replace({
    'Ground': 0, 'ground': 0, 'Ground Floor': 0
})

num_cols = ['Floor','Year Built','Price','Area','Bedrooms','Bathrooms']
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

# ==========================================
# 🚨 7. REMOVE ROWS WITHOUT TARGET
# ==========================================
df = df[df['Price'].notna()]
df = df[df['Price'] > 0]

df['Log_Price'] = np.log1p(df['Price'])

# ==========================================
# 🔥 8. RF IMPUTATION: FLOOR (NO PRICE)
# ==========================================
print("Imputing Floor without Leakage...")

# شيلنا السعر من الـ features
floor_features = ['Area', 'Bedrooms', 'Bathrooms']

for c in floor_features:
    df[c] = df[c].fillna(df[c].median())

floor_train = df[df['Floor'].notna()]
floor_missing = df[df['Floor'].isna()]

rf_floor = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

if len(floor_missing) > 0:
    rf_floor.fit(floor_train[floor_features], floor_train['Floor'])
    df.loc[df['Floor'].isna(), 'Floor'] = rf_floor.predict(floor_missing[floor_features])

df['Floor'] = df['Floor'].astype(int)

# ==========================================
# 🔥 9. RF IMPUTATION: YEAR BUILT (NO PRICE)
# ==========================================
print("Imputing Year Built without Leakage...")

# شيلنا السعر واعتمدنا على المساحة والمواصفات والدور
year_features = ['Area', 'Bedrooms', 'Bathrooms', 'Floor']

for c in year_features:
    df[c] = df[c].fillna(df[c].median())

year_train = df[df['Year Built'].notna()]
year_missing = df[df['Year Built'].isna()]

rf_year = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

if len(year_missing) > 0:
    rf_year.fit(year_train[year_features], year_train['Year Built'])
    df.loc[df['Year Built'].isna(), 'Year Built'] = rf_year.predict(year_missing[year_features])

df['Year Built'] = df['Year Built'].astype(int)

# ==========================================
# 10. Fill remaining numeric
# ==========================================
for c in ['Area','Bedrooms','Bathrooms']:
    df[c] = df[c].fillna(df[c].median())

# ==========================================
# 11. Smart Finish Imputation (Random Forest Classifier)
# ==========================================
print("Imputing Finish using RandomForestClassifier...")

# استبدال Unknown بـ NaN عشان نعوضها صح
df['Finish'] = df['Finish'].replace('Unknown', np.nan)

# هنستخدم الخصائص دي بس عشان نتوقع التشطيب (بدون السعر لمنع الـ Leakage)
finish_features = ['Area', 'Bedrooms', 'Bathrooms', 'Floor']

# التأكد إن مفيش Nulls في الخصائص دي (إحنا كده كده عوضناها فوق)
finish_train = df[df['Finish'].notna()]
finish_missing = df[df['Finish'].isna()]

# استخدام Classifier مع موازنة الفئات (السر كله في class_weight)
rf_finish = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight='balanced',  # <--- هو ده السحر اللي هيحل المشكلة
    n_jobs=-1
)

if len(finish_missing) > 0:
    # تدريب الموديل على الشقق اللي تشطيبها معروف
    rf_finish.fit(finish_train[finish_features], finish_train['Finish'])
    
    # توقع التشطيب للشقق اللي تشطيبها ناقص
    df.loc[df['Finish'].isna(), 'Finish'] = rf_finish.predict(finish_missing[finish_features])

# لو اتبقى أي حاجة نادرة جداً نعوضها بنص تشطيب كحل أخير آمن
df['Finish'] = df['Finish'].fillna('Semi Finished')

print("\nFinish distribution after RF Imputation:")
print(df['Finish'].value_counts())

# ==========================================
# 12. Categorical cleaning
# ==========================================
df['Location'] = df['Location'].fillna('Unknown')
df['District'] = df['District'].fillna('Unknown')

# ==========================================
# 13. Amenities safe encoding (NO PRICE IN LOGIC)
# ==========================================
amenities = [
    'Furnished','Electricity Meter','Natural Gas',
    'Water Meter','Elevator','Security'
]

for c in amenities:
    df[c] = df[c].map({'Yes':1,'No':0}).fillna(0)

# شيلنا السعر من منطق الأثاث لأنه Leakage، خلينا التعويض كله بأصفار أو بناءً على معطيات تانية
# لو مساحة الشقة كبيرة جداً والتشطيب عالي ممكن نفترض إنها مفروشة، بس الأفضل نسيبها 0 لو مفيش داتا 
df['Furnished'] = df['Furnished'].fillna(0)

df['Elevator'] = np.where(
    (df['Elevator'].isna()) & (df['Floor'] > 5),
    1,
    df['Elevator']
)

df[amenities] = df[amenities].fillna(0).astype(int)

# ==========================================
# 🚨 14. FINAL SAFETY CHECK
# ==========================================
df = df.dropna(subset=['Log_Price'])

# ==========================================
# 15. Train/Test split
# ==========================================
X = df.drop(columns=['Price','Log_Price'])
y = df['Log_Price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# ==========================================
# 16. Target Encoding
# ==========================================
categorical_cols = ['Location','District','Finish']

encoder = ColumnTransformer(
    [('target', TargetEncoder(random_state=42), categorical_cols)],
    remainder='passthrough'
)

X_train_encoded = encoder.fit_transform(X_train, y_train)
X_test_encoded = encoder.transform(X_test)

# ==========================================
# 17. Convert to DataFrame safely
# ==========================================
cols = encoder.get_feature_names_out()

X_train_encoded = pd.DataFrame(X_train_encoded, columns=cols)
X_test_encoded = pd.DataFrame(X_test_encoded, columns=cols)

print("✅ Pipeline Ready without Data Leakage!")
print("Train shape:", X_train_encoded.shape)
print("Test shape:", X_test_encoded.shape)

Imputing Floor without Leakage...
Imputing Year Built without Leakage...
Imputing Finish using RandomForestClassifier...

Finish distribution after RF Imputation:
Finish
Super Lux          12575
Lux                 2996
Extra Super Lux     2549
Without Finish      1283
Semi Finished       1169
Name: count, dtype: int64
✅ Pipeline Ready without Data Leakage!
Train shape: (16457, 14)
Test shape: (4115, 14)


In [2]:
# حفظ الداتا بعد المعالجة
output_path = 'Processed_Properties.xlsx'
df.to_excel(output_path, index=False)

print(f"Data successfully saved to {output_path}")

Data successfully saved to Processed_Properties.xlsx


In [3]:
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    ),
    'CatBoost': CatBoostRegressor(
        iterations=2000,
        learning_rate=0.03,
        depth=8,
        loss_function='RMSE',
        random_seed=42,
        verbose=0
    )
}

print("=== Model Evaluation and Comparison ===\n")

results = []

best_model = None
best_model_name = None
best_rmse = float("inf")

y_test_actual = np.expm1(y_test)

# ==========================================
# 🔥 TRAIN LOOP (FIXED)
# ==========================================
for name, model in models.items():

    # =========================
    # Fit handling per model
    # =========================
    if name in ['XGBoost', 'CatBoost']:
        model.fit(
            X_train_encoded,
            y_train,
            eval_set=[(X_test_encoded, y_test)],
            verbose=False
        )
    else:
        model.fit(X_train_encoded, y_train)

    # =========================
    # Predict
    # =========================
    y_pred_log = model.predict(X_test_encoded)
    y_pred_actual = np.expm1(y_pred_log)

    # =========================
    # Metrics
    # =========================
    mae = mean_absolute_error(y_test_actual, y_pred_actual)
    rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))
    r2 = r2_score(y_test_actual, y_pred_actual)

    print(f"--- {name} ---")
    print(f"MAE:  {mae:,.0f} EGP")
    print(f"RMSE: {rmse:,.0f} EGP")
    print(f"R²: {r2:.4f}\n")

    # best model
    if rmse < best_rmse:
        best_rmse = rmse
        best_model = model
        best_model_name = name

print(f"🏆 Best Model Selected: {best_model_name}")

=== Model Evaluation and Comparison ===

--- Linear Regression ---
MAE:  559,591 EGP
RMSE: 940,745 EGP
R²: 0.7345

--- Ridge Regression ---
MAE:  559,486 EGP
RMSE: 940,449 EGP
R²: 0.7346

--- Random Forest ---
MAE:  429,270 EGP
RMSE: 705,789 EGP
R²: 0.8505

--- XGBoost ---
MAE:  427,987 EGP
RMSE: 698,222 EGP
R²: 0.8537

--- CatBoost ---
MAE:  429,906 EGP
RMSE: 702,903 EGP
R²: 0.8518

🏆 Best Model Selected: XGBoost


In [4]:
best_model.fit(X_train_encoded, y_train)
print("✅ Final model trained successfully!")

# ==========================================
# 19. Function to Predict Price for New Property
# ==========================================
def predict_property_price(property_data, encoder, model, feature_columns):
    """
    Predicts the property price based on input features.

    Parameters:
        property_data (dict): Dictionary containing property features.
        encoder: Fitted ColumnTransformer with TargetEncoder.
        model: Trained regression model.
        feature_columns (list): Columns used during training.

    Returns:
        float: Predicted price in EGP.
    """
    # تحويل البيانات إلى DataFrame
    input_df = pd.DataFrame([property_data])

    # التأكد من وجود جميع الأعمدة
    for col in feature_columns:
        if col not in input_df.columns:
            input_df[col] = 0

    # ترتيب الأعمدة كما في التدريب
    input_df = input_df[feature_columns]

    # تطبيق نفس الـ Encoding
    input_encoded = encoder.transform(input_df)
    input_encoded = pd.DataFrame(
        input_encoded,
        columns=encoder.get_feature_names_out()
    )

    # التنبؤ (في مقياس اللوغاريتم)
    pred_log = model.predict(input_encoded)

    # تحويل النتيجة إلى السعر الحقيقي
    predicted_price = np.expm1(pred_log)[0]

    return predicted_price

✅ Final model trained successfully!


In [5]:
import joblib

# حفظ أفضل موديل (XGBoost)
joblib.dump(best_model, 'xgboost_real_estate_model.pkl')

# حفظ الـ Encoder (مهم جداً عشان تحويل أسماء المناطق لنفس الأرقام)
joblib.dump(encoder, 'data_encoder.pkl')

print("Model and Encoder saved successfully!")

Model and Encoder saved successfully!


In [6]:
# مثال لعقار جديد
new_property = {
    'Area': 85,
    'Bedrooms': 2,
    'Bathrooms': 1,
    'Location': 'Abu Youssef',
    'District': 'El Agami',
    'Year Built': 2010,
    'Finish': 'Semi Finished',
    'Floor': 4,
    'Furnished': 0,
    'Electricity Meter': 0,
    'Natural Gas': 1,
    'Water Meter': 1,
    'Elevator': 0,
    'Security': 1
}

predicted_price = predict_property_price(
    new_property,
    encoder,
    best_model,
    X_train.columns
)

print(f"🏠 Predicted Property Price: {predicted_price:,.0f} EGP")

🏠 Predicted Property Price: 473,945 EGP


In [7]:
# 330000	85	2	1	Abu Youssef	El Agami	2010	Semi Finished	4	0	0	0	0	0	0	0	12.70685096

In [8]:
# ==========================================
# 7. تجربة الموديل على عينة عشوائية (مع إضافة Location وتنسيق احترافي)
# ==========================================
import pandas as pd
import numpy as np

print("=== Random Sample Predictions ===")

# سحب 20 صفاً عشوائياً من بيانات الاختبار
sample_X = X_test.sample(n=20, random_state=np.random.randint(1, 20000))
sample_y = y_test.loc[sample_X.index]

# عمل Encoding للعينة بنفس الطريقة المستخدمة في التدريب
sample_X_encoded = pd.DataFrame(
    encoder.transform(sample_X), 
    columns=encoder.get_feature_names_out(),
    index=sample_X.index
)

# التوقع باستخدام أفضل موديل (CatBoost)
sample_preds_log = best_model.predict(sample_X_encoded)

# إرجاع الأسعار لأصلها (عكس اللوغاريتم)
actual_prices = np.expm1(sample_y)
predicted_prices = np.expm1(sample_preds_log)

# تجميع النتائج في DataFrame للعرض (إضافة Location بعد District)
results_df = pd.DataFrame({
    'District': sample_X['District'],
    'Location': sample_X['Location'], # العمود الجديد هنا
    'Area (m²)': sample_X['Area'],
    'Bedrooms': sample_X['Bedrooms'],
    'Finish': sample_X['Finish'],
    'Actual Price': actual_prices.values,
    'Predicted Price': predicted_prices,
})

# حساب الفرق المادي ونسبة الخطأ المئوية
results_df['Difference'] = results_df['Predicted Price'] - results_df['Actual Price']
results_df['Error %'] = (abs(results_df['Difference']) / results_df['Actual Price']) * 100

# ==========================================
# 🎨 تنسيق الجدول بشكل احترافي لـ Jupyter Notebook
# ==========================================
styled_df = (
    results_df.style
    .format({
        'Area (m²)': '{:.0f}',
        'Bedrooms': '{:.0f}',
        'Actual Price': '{:,.0f} EGP',
        'Predicted Price': '{:,.0f} EGP',
        'Difference': '{:+,.0f} EGP', 
        'Error %': '{:.2f}%'
    })
    # تدرج لوني لعمود نسبة الخطأ
    .background_gradient(subset=['Error %'], cmap='RdYlGn_r')
    
    # رسم بياني صغير جوه عمود الفرق
    .bar(subset=['Difference'], align='mid', color=['#ff9999', '#99ff99'])
    
    # تنسيق الخطوط وتوسيط النص
    .set_properties(**{'text-align': 'center', 'font-size': '13px', 'border': '1px solid #d3d3d3'})
    
    # تنسيق رأس الجدول
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#2c3e50'), 
                                     ('color', 'white'), 
                                     ('font-size', '14px'), 
                                     ('text-align', 'center'),
                                     ('border', '1px solid white')]},
        {'selector': 'caption', 'props': [('font-size', '18px'), ('font-weight', 'bold'), ('margin-bottom', '10px')]}
    ])
    .set_caption("🏠 Detailed Property Price Predictions (Random Sample) 🏠")
    .hide(axis="index")
)

# عرض الجدول المنسق
styled_df

=== Random Sample Predictions ===


District,Location,Area (m²),Bedrooms,Finish,Actual Price,Predicted Price,Difference,Error %
El Agami,Abu Youssef,125,3,Super Lux,"1,500,000 EGP","1,149,948 EGP","-350,052 EGP",23.34%
Wasat,Bakoos,175,3,Super Lux,"2,250,000 EGP","2,639,100 EGP","+389,100 EGP",17.29%
El Agami,El Hanouvel,90,2,Lux,"650,000 EGP","436,207 EGP","-213,793 EGP",32.89%
El Agami,El Bitash,90,2,Lux,"490,000 EGP","619,942 EGP","+129,942 EGP",26.52%
El Montazah,Sidi Bishr,120,3,Super Lux,"1,800,000 EGP","1,978,782 EGP","+178,782 EGP",9.93%
El Agami,El Hanouvel,80,2,Super Lux,"400,000 EGP","407,892 EGP","+7,892 EGP",1.97%
El Agami,El Bitash,120,2,Semi Finished,"1,800,000 EGP","866,662 EGP","-933,338 EGP",51.85%
El Montazah,Sidi Bishr,200,3,Extra Super Lux,"2,800,000 EGP","3,715,989 EGP","+915,989 EGP",32.71%
El Agami,El Hanouvel,130,3,Super Lux,"1,600,000 EGP","735,532 EGP","-864,468 EGP",54.03%
El Montazah,Sidi Bishr,110,3,Semi Finished,"2,860,000 EGP","1,966,394 EGP","-893,606 EGP",31.24%
